In [1]:
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

In [3]:
def partition_document(file_path):
    elements=partition_pdf(
        filename=file_path,
        strategy='hi_res',
        infer_table_structure=True,
        extract_image_block_types=['Image'],
        extract_image_block_to_payload=True
        
    )
    return elements

file_path='/home/obs/Desktop/RAG/RAG_initial/notebooks/doc/attention-is-all-you-need-1.pdf'
elements=partition_pdf(file_path)

No languages specified, defaulting to English.


In [5]:
def create_chunks(elements):
    chunks=chunk_by_title(
        elements=elements,
        max_characters=3000,
        new_after_n_chars=2400,
        combine_text_under_n_chars=500
    )
    return chunks

chunks=create_chunks(elements)

In [16]:
import json
from typing import List

def seperate_content_types(chunk):
    content_data = {
        "text": chunk.text,
        "tables": [],
        "images": [],
        "types": ["text"]
    }

    if hasattr(chunk, "metadata") and hasattr(chunk.metadata, "orig_elements"):
        for element in chunk.metadata.orig_elements:

            content_type = type(element).__name__

            if content_type == "Table":
                content_data["types"].append("table")

                table_html = getattr(
                    element.metadata,
                    "text_as_html",
                    element.text
                )

                content_data["tables"].append(table_html)

            elif content_type == "Image":
                if (
                    hasattr(element, "metadata")
                    and hasattr(element.metadata, "image_base64")
                ):
                    content_data["types"].append("image")
                    content_data["images"].append(
                        element.metadata.image_base64
                    )

    content_data["types"] = list(set(content_data["types"]))

    return content_data


def create_ai_summary(
    text: str,
    tables: List[str],
    images: List[str]
):
    try:
        llm = ChatOpenAI(
            model="gpt-4o",
            temperature=0
        )

        prompt_text = f"""
You are creating a searchable description for document content retrieval.

CONTENT TO ANALYZE:

TEXT CONTENT:
{text}
"""

        if tables:
            prompt_text += "\nTABLES:\n"

            for i, table in enumerate(tables):
                prompt_text += f"""
Table {i+1}:
{table}

"""

        prompt_text += """
YOUR TASK:

Generate a comprehensive, searchable description that covers:

1. Key facts, numbers, and data points from text and tables
2. Main topics and concepts discussed
3. Questions this content could answer
4. Visual content analysis
5. Alternative search terms users might use

Make it detailed and searchable.

SEARCHABLE DESCRIPTION:
"""

        message_content = [
            {
                "type": "text",
                "text": prompt_text
            }
        ]

        for image_base64 in images:
            message_content.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{image_base64}"
                }
            })

        message = HumanMessage(
            content=message_content
        )

        response = llm.invoke([message])

        return response.content

    except Exception as e:
        print(f"AI summary failed: {e}")

        summary = f"{text[:300]}..."

        if tables:
            summary += f" [Contains {len(tables)} table(s)]"

        if images:
            summary += f" [Contains {len(images)} image(s)]"

        return summary


def create_summary(chunks):
    langchain_documents = []

    for i, chunk in enumerate(chunks):

        print(f"Processing chunk {i+1}/{len(chunks)}")

        content_data = seperate_content_types(chunk)

        if content_data["tables"] or content_data["images"]:

            print(
                f"Sending mixed content for summarization: "
                f"{content_data['types']}"
            )

            try:
                enhanced_content = create_ai_summary(
                    content_data["text"],
                    content_data["tables"],
                    content_data["images"]
                )

                print("AI summary created successfully")
                print(
                    f"Preview: {enhanced_content[:200]}..."
                )

            except Exception as e:
                print(f"AI summary failed: {e}")
                enhanced_content = content_data["text"]

        else:
            print("Using raw text (no tables/images)")
            enhanced_content = content_data["text"]

        doc = Document(
            page_content=enhanced_content,
            metadata={
                "original_content": json.dumps({
                    "raw_text": content_data["text"],
                    "tables_html": content_data["tables"],
                    "images_base64": content_data["images"]
                })
            }
        )

        langchain_documents.append(doc)

    print(
        f"✅ Processed {len(langchain_documents)} chunks"
    )

    return langchain_documents


processed_chunks = create_summary(chunks)

Processing chunk 1/33
Using raw text (no tables/images)
Processing chunk 2/33
Using raw text (no tables/images)
Processing chunk 3/33
Using raw text (no tables/images)
Processing chunk 4/33
Using raw text (no tables/images)
Processing chunk 5/33
Using raw text (no tables/images)
Processing chunk 6/33
Using raw text (no tables/images)
Processing chunk 7/33
Using raw text (no tables/images)
Processing chunk 8/33
Using raw text (no tables/images)
Processing chunk 9/33
Using raw text (no tables/images)
Processing chunk 10/33
Using raw text (no tables/images)
Processing chunk 11/33
Using raw text (no tables/images)
Processing chunk 12/33
Using raw text (no tables/images)
Processing chunk 13/33
Using raw text (no tables/images)
Processing chunk 14/33
Using raw text (no tables/images)
Processing chunk 15/33
Using raw text (no tables/images)
Processing chunk 16/33
Using raw text (no tables/images)
Processing chunk 17/33
Using raw text (no tables/images)
Processing chunk 18/33
Using raw text (n

In [34]:
def create_vector_store(documents, persist_directory):
    print('createing chrome db......')
    
    embadding_mode=OpenAIEmbeddings(model='text-embedding-3-small')
    
    vectorstore=Chroma.from_documents(
        documents=documents,
        persist_directory=persist_directory,
        embedding=embadding_mode,
        collection_metadata={"hns:space":'cosine'}
    )
    
    print('Vectore store is created ...........')
    return vectorstore

persist_directory='/home/obs/Desktop/RAG/RAG_initial/notebooks/db/chrome'
db=create_vector_store(processed_chunks, persist_directory)

createing chrome db......
Vectore store is created ...........


In [ ]:
query='What are the two main components of the Transformer architecture? give me the answer it should be in max 50 chars, dont give the space inbetween  '
retriver=db.as_retriever(search_kwargs={'k':3})
chunks=retriver.invoke(query)

print(chunks)

[Document(id='4f9ec046-af75-4a18-9ad6-f3696b482442', metadata={'original_content': '{"raw_text": "Figure 1: The Transformer - model architecture.\\n\\nThe Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, respectively.\\n\\n3.1 Encoder and Decoder Stacks\\n\\nEncoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position- wise fully connected feed-forward network. We employ a residual connection [11] around each of the two sub-layers, followed by layer normalization [1]. That is, the output of each sub-layer is LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding layers, produc

In [ ]:
for i in chunks:
    print(i)
    print('*'*200)

In [50]:
def generate_final_answer(chunks, query):
    """Generate final answer using multimodal content"""
    
    try:
        # Initialize LLM (needs vision model for images)
        llm = ChatOpenAI(model="gpt-4o", temperature=0)
        
        # Build the text prompt
        prompt_text = f"""Based on the following documents, please answer this question: {query}

CONTENT TO ANALYZE:
"""
        
        for i, chunk in enumerate(chunks):
            prompt_text += f"--- Document {i+1} ---\n"
            
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                
                # Add raw text
                raw_text = original_data.get("raw_text", "")
                if raw_text:
                    prompt_text += f"TEXT:\n{raw_text}\n\n"
                
                # Add tables as HTML
                tables_html = original_data.get("tables_html", [])
                if tables_html:
                    prompt_text += "TABLES:\n"
                    for j, table in enumerate(tables_html):
                        prompt_text += f"Table {j+1}:\n{table}\n\n"
            
            prompt_text += "\n"
        
        prompt_text += """
Please provide a clear, comprehensive answer using the text, tables, and images above. If the documents don't contain sufficient information to answer the question, say "I don't have enough information to answer that question based on the provided documents."

ANSWER:"""

        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]
        
        # Add all images from all chunks
        for chunk in chunks:
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                images_base64 = original_data.get("images_base64", [])
                
                for image_base64 in images_base64:
                    message_content.append({
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
                    })
        
        # Send to AI and get response
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        
        return response.content
        
    except Exception as e:
        print(f"❌ Answer generation failed: {e}")
        return "Sorry, I encountered an error while generating the answer."

# Usage
final_answer = generate_final_answer(chunks, query)
print(final_answer)



Encoder, Decoder
